# 1 · Download GPM IMERG half-hourly files

Downloads the IMERG V07 half-hourly files needed to build 3 or 6 hourly rainfall totals
for the periods listed in `examples/periods.csv`.

You need a free **NASA Earthdata** account: <https://urs.earthdata.nasa.gov/users/new>.
Each file is about 8 MB (≈ 400 MB per day of data).

In [ ]:
# Cell 1 – install packages (run once, then comment out)
# !pip install -r ../requirements.txt

In [1]:
# Cell 2 – settings
import os, sys
sys.path.insert(0, os.path.abspath('..'))   # so the notebook can import imerg_utils.py from the repo root
import pandas as pd
import imerg_utils as iu

import earthaccess

PERIODS_CSV      = '../examples/periods.csv'  # name,start,end – edit or point to your own file
HOURS            = 6         # accumulation length: 3 or 6
UTC_OFFSET_HOURS = 0         # 0 if the periods are in UTC; 6 if they are Bangladesh time, etc.
PRODUCT          = 'final'   # 'final' (gauge-adjusted), 'late' or 'early'
DATA_DIR         = '../data/imerg'   # where the HDF5 files are saved

periods = pd.read_csv(PERIODS_CSV)
periods

,name,start,end
0,demo_2024_07_19,2024-07-19 00:00,2024-07-20 06:00


In [2]:
# Cell 3 – work out which files each period needs
# A row labelled 06:00 is the rain from 00:00 to 06:00, so each period needs files
# starting from (first row - HOURS) up to the last row.
plan = []
for p in periods.itertuples():
    bins = iu.make_bin_ends(p.start, p.end, HOURS, UTC_OFFSET_HOURS)
    first, last = iu.download_window(bins, HOURS)
    plan.append({'period': p.name, 'bins': len(bins), 'first_bin_end_utc': bins[0], 'last_bin_end_utc': bins[-1],
                 'download_from_utc': first, 'download_to_utc': last,
                 'files': len(iu.required_file_times(bins, HOURS))})
plan = pd.DataFrame(plan)
print(f"About {plan.files.sum() * 8 / 1024:.1f} GB to download")
plan

About 0.5 GB to download


,period,bins,first_bin_end_utc,last_bin_end_utc,download_from_utc,download_to_utc,files
0,demo_2024_07_19,5,2024-07-19 06:00:00,2024-07-20 06:00:00,2024-07-19,2024-07-20 06:00:00,60


In [3]:
# Cell 4 – Earthdata login (asks for username/password the first time; persist=True saves them in ~/.netrc)
auth = earthaccess.login(persist=True)
print('Authenticated:', auth.authenticated)

Enter your Earthdata Login username:  arnob008
Enter your Earthdata password:  ········


Authenticated: True


In [4]:
# Cell 5 – download (safe to re-run: files already in DATA_DIR are skipped)
for p in plan.itertuples():
    print(f'--- {p.period}')
    iu.download_imerg(p.download_from_utc, p.download_to_utc, DATA_DIR, product=PRODUCT)

--- demo_2024_07_19
2024-07-19 00:00:00 -> 2024-07-20 06:00:00: 60 files found, 60 expected


QUEUEING TASKS | :   0%|          | 0/60 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/60 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/60 [00:00<?, ?it/s]

In [5]:
# Cell 6 – check that every needed file is on disk
have = iu.index_imerg_files(DATA_DIR)
for p in periods.itertuples():
    need = iu.required_file_times(iu.make_bin_ends(p.start, p.end, HOURS, UTC_OFFSET_HOURS), HOURS)
    missing = [t for t in need if t not in have]
    status = 'OK' if not missing else f'MISSING {len(missing)} (first: {missing[0]}) – re-run cell 5'
    print(f'{p.name}: {len(need) - len(missing)}/{len(need)} files  {status}')

demo_2024_07_19: 60/60 files  OK
